# 08b. Preprocesamiento de Datos y Preparación para el Modelado (Alternativa B)

Este notebook corresponde a la **Fase 3: Tarea 3.1 - Preprocesamiento para la Alternativa B (Muestreo Espacio-Temporal Puro)**. 
En esta etapa prepararemos la parrilla de salida para nuestro modelo predictivo campeón (LightGBM) bajo el nuevo muestreo.

### Justificación Metodológica: ¿Por qué no normalizamos ni escalamos?
A diferencia de los modelos lineales o basados en distancias (como SVM o K-NN), los algoritmos basados en árboles de decisión (como Random Forest, XGBoost y LightGBM) son **invariantes ante transformaciones monótonas** de las variables de entrada. 
Esto significa que dividir una variable por su máximo o restarle su media no altera los puntos de corte óptimos que el árbol encuentra para segmentar los datos.

Evitando el escalado (como *StandardScaler* o *MinMaxScaler*):
1. **Preservamos la interpretabilidad física directa:** Las variables retienen sus unidades físicas originales (grados de pendiente, metros de altitud, km/h de velocidad de viento, ºC de temperatura). Esto es extremadamente valioso para la memoria del TFG y la posterior interpretabilidad del modelo (SHAP).
2. **Evitamos riesgo de data leakage:** No introducimos estadísticas globales de test en el conjunto de entrenamiento.
3. **Eficiencia y sencillez:** Menos pasos en el pipeline de preprocesamiento.

## 1. Carga de Librerías y Carga de Incendios Reales y Ausencias

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path


BASE_DIR = Path("..")
INCENDIOS_PATH = BASE_DIR / "datos" / "processed" / "07_dataset_final_TOPOGRAFICO.csv"
AUSENCIAS_PATH = BASE_DIR / "datos" / "processed" / "03_ausencias_topografia_altB.csv"
OUT_DIR = BASE_DIR / "datos" / "processed"

print("Cargando dataset de incendios reales (Positivos)...")
df_pos_raw = pd.read_csv(INCENDIOS_PATH, sep=';', decimal=',')
print(f"Dimensiones incendios brutos: {df_pos_raw.shape}")

print("Cargando dataset de ausencias space-time (Negativos)...")
df_neg = pd.read_csv(AUSENCIAS_PATH, sep=';', decimal=',')
print(f"Dimensiones ausencias: {df_neg.shape}")

Cargando dataset de incendios reales (Positivos)...
Dimensiones incendios brutos: (4955, 29)
Cargando dataset de ausencias space-time (Negativos)...
Dimensiones ausencias: (4476, 18)


## 2. Limpieza de Incendios Reales (Positivos) y Unificación del Dataset Balanceado

Para que el dataset esté perfectamente balanceado en proporción 50/50 y libre de duplicados clónicos, aplicaremos el mismo filtro de unicidad y mapeo de superficies corruptas que en la Alternativa A sobre los incendios reales.

In [2]:
MAPEO_SUPERFICIES_CORRUPTAS = {
    "1.501.393": 150.1393,
    "10.699.999.999.999.900": 1.07,
    "11.904.680.000.000.000": 11904.68,
    "116.044.237": 11604.4237,
    "12.000.000.000.000.000": 1.2,
    "12.999.999.999.999.900": 1.3,
    "14.300.000.000.000.000": 1.43,
    "14.388.999.999.999.900": 14.389,
    "14.626.675": 1462.6675,
    "15.000.000.000.000.000": 1.5,
    "17.000.000.000.000.000": 1.7,
    "17.939.999.999.999.900": 17.94,
    "185.358.806": 18535.8806,
    "21.500.000.000.000.000": 2.15,
    "26.649.999.999.999.900": 266.5,
    "30.501.737.000.000.000": 3050.1737,
    "4.732.155": 473.2155,
    "4.752.199.999.999.990": 475.22,
    "6.255.800.000.000.000": 625.58,
    "60.600.000.000.000.000": 6.06,
    "65.440.000.000.000.000": 6.544,
    "7.100.699.999.999.990": 710.07,
    "7.333.999.999.999.990": 733.4,
    "7.399.999.999.999.990": 7.4,
    "7.459.900.000.000.000": 745.99,
    "7.791.000.000.000.000": 77.91,
    "8.094.999.999.999.990": 80.95,
    "8.153.999.999.999.990": 81.54,
    "8.185.011.000.000.000": 818.5011,
    "8.319.000.000.000.000": 83.19,
}

df_pos = df_pos_raw.copy()
df_pos['Superficie_Total_Real'] = df_pos['Superficie_Total_Real'].astype(str).str.strip().replace(MAPEO_SUPERFICIES_CORRUPTAS)

cols_numericas = ['lat', 'lon', 'Superficie_Total_Real']
for col in cols_numericas:
    df_pos[col] = pd.to_numeric(df_pos[col].astype(str).str.replace(',', '.'), errors='coerce')

df_pos_limpio = df_pos.drop_duplicates(subset=['lat', 'lon', 'fecha_ini', 'Superficie_Total_Real']).copy()
df_pos_limpio['fecha_dt'] = pd.to_datetime(df_pos_limpio['fecha_ini'])
df_pos_limpio = df_pos_limpio.sort_values('fecha_dt').reset_index(drop=True)
df_pos_limpio = df_pos_limpio.drop(columns=['fecha_dt'])

print(f"Incendios reales limpios: {df_pos_limpio.shape[0]}")

cols_to_keep = [
    'lat', 'lon', 'fecha_ini', 'target', 'Superficie_Total_Real',
    'tipo_vegetacion', 'elevacion', 'pendiente', 'orientacion',
    'temp_max', 'temp_media', 'temp_min', 'precipitacion',
    'viento_medio', 'racha_max', 'dir_viento', 'humedad_media',
    'prec_acum_7d', 'tmax_max_7d', 'dias_sin_lluvia'
]

df_pos_limpio['target'] = 1
df_neg['target'] = 0
df_neg['Superficie_Total_Real'] = 0.0

df_pos_final = df_pos_limpio[[c for c in cols_to_keep if c in df_pos_limpio.columns]].copy()
df_neg_final = df_neg[[c for c in cols_to_keep if c in df_neg.columns]].copy()

df_balanced = pd.concat([df_pos_final, df_neg_final], ignore_index=True)
df_balanced = df_balanced[cols_to_keep]

BAL_DATA_PATH = OUT_DIR / "08_dataset_modelado_BALANCEADO_altB.csv"
df_balanced.to_csv(BAL_DATA_PATH, sep=';', index=False, decimal=',')
print(f"Dataset balanceado Alt B guardado con éxito en: {BAL_DATA_PATH.name}")
print(f"Proporción de target:\n{df_balanced['target'].value_counts()}")

Incendios reales limpios: 4476
Dataset balanceado Alt B guardado con éxito en: 08_dataset_modelado_BALANCEADO_altB.csv
Proporción de target:
target
1    4476
0    4476
Name: count, dtype: int64


## 3. Preprocesamiento e Ingeniería de Características

Aplicaremos la misma normalización categórica para la vegetación y OHE para asegurar que las variables dummy coincidan exactamente en orden y cantidad con la Alternativa A.

In [3]:
cols_numericas_balanced = [
    'lat', 'lon', 'Superficie_Total_Real', 'elevacion', 'pendiente', 'orientacion',
    'temp_max', 'temp_media', 'temp_min', 'precipitacion', 'viento_medio', 'racha_max',
    'dir_viento', 'humedad_media', 'prec_acum_7d', 'tmax_max_7d', 'dias_sin_lluvia'
]

for col in cols_numericas_balanced:
    if col in df_balanced.columns:
        if df_balanced[col].dtype == 'object':
            df_balanced[col] = pd.to_numeric(df_balanced[col].astype(str).str.replace(',', '.'), errors='coerce')
        else:
            df_balanced[col] = pd.to_numeric(df_balanced[col], errors='coerce')

df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].astype(str).str.strip()
df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].str.replace(r'.*Con.*feras.*', 'Coniferas', regex=True)
df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].str.replace(r'.*Agr.*cola.*', 'Agricola', regex=True)
df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].str.replace(r'.*Matorral.*', 'Matorral', regex=True)
df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].str.replace(r'.*Pastizal.*', 'Pastizal', regex=True)
df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].str.replace(r'.*Frondosas.*', 'Frondosas', regex=True)
df_balanced['tipo_vegetacion'] = df_balanced['tipo_vegetacion'].str.replace(r'.*Urbano/Otros.*', 'Urbano_Otros', regex=True)


print("Categorías de vegetación:")
print(df_balanced['tipo_vegetacion'].value_counts())

df_dummies = pd.get_dummies(df_balanced['tipo_vegetacion'], prefix='veg', prefix_sep='_', drop_first=False)
df_preprocessed = pd.concat([df_balanced.drop(columns=['tipo_vegetacion']), df_dummies], axis=1)

expected_dummies = ['veg_Agricola', 'veg_Coniferas', 'veg_Frondosas', 'veg_Matorral', 'veg_Pastizal', 'veg_Urbano_Otros']
for dummy in expected_dummies:
    if dummy not in df_preprocessed.columns:
        df_preprocessed[dummy] = False

if df_preprocessed['dir_viento'].dtype == 'object':
    df_preprocessed['dir_viento'] = pd.Categorical(df_preprocessed['dir_viento']).codes

print(f"\nColumnas resultantes: {df_preprocessed.shape[1]}")

Categorías de vegetación:
tipo_vegetacion
Coniferas             2602
Matorral              2251
Agricola              2002
Frondosas              895
Urbano_Otros           698
Pastizal               497
Urbano_Antropizado       7
Name: count, dtype: int64

Columnas resultantes: 26


## 4. Partición Train/Test y Exportación de Matrices de la Alternativa B

In [4]:
cols_eliminar = ['target', 'Superficie_Total_Real', 'fecha_ini']
cols_existentes_eliminar = [col for col in cols_eliminar if col in df_preprocessed.columns]

X_cols_ordered = [
    'lat', 'lon', 'elevacion', 'pendiente', 'orientacion', 'temp_max', 'temp_media', 'temp_min',
    'precipitacion', 'viento_medio', 'racha_max', 'dir_viento', 'humedad_media', 'prec_acum_7d',
    'tmax_max_7d', 'dias_sin_lluvia', 'veg_Agricola', 'veg_Coniferas', 'veg_Frondosas',
    'veg_Matorral', 'veg_Pastizal', 'veg_Urbano_Antropizado', 'veg_Urbano_Otros'
]

X = df_preprocessed[X_cols_ordered]
y = df_preprocessed['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print("===========================================================")
print(" ESTRUCTURA DE PARTICIÓN DE DATOS (TRAIN / TEST SPLIT - ALT B)")
print("===========================================================")
print(f"Matriz de Entrenamiento (X_train_altB) : {X_train.shape[0]} filas x {X_train.shape[1]} columnas")
print(f"Matriz de Prueba (X_test_altB)        : {X_test.shape[0]} filas x {X_test.shape[1]} columnas")
print(f"Vector de Entrenamiento (y_train_altB): {y_train.shape[0]} registros")
print(f"Vector de Prueba (y_test_altB)        : {y_test.shape[0]} registros")
print("===========================================================")

X_TRAIN_PATH = OUT_DIR / "X_train_altB.csv"
X_TEST_PATH = OUT_DIR / "X_test_altB.csv"
Y_TRAIN_PATH = OUT_DIR / "y_train_altB.csv"
Y_TEST_PATH = OUT_DIR / "y_test_altB.csv"

X_train.to_csv(X_TRAIN_PATH, sep=';', index=False, decimal=',')
X_test.to_csv(X_TEST_PATH, sep=';', index=False, decimal=',')
pd.DataFrame(y_train).to_csv(Y_TRAIN_PATH, sep=';', index=False, decimal=',')
pd.DataFrame(y_test).to_csv(Y_TEST_PATH, sep=';', index=False, decimal=',')
print("¡Matrices exportadas con éxito!")

 ESTRUCTURA DE PARTICIÓN DE DATOS (TRAIN / TEST SPLIT - ALT B)
Matriz de Entrenamiento (X_train_altB) : 7161 filas x 23 columnas
Matriz de Prueba (X_test_altB)        : 1791 filas x 23 columnas
Vector de Entrenamiento (y_train_altB): 7161 registros
Vector de Prueba (y_test_altB)        : 1791 registros
¡Matrices exportadas con éxito!
